# Hasem — Accident Analysis System v2
**YOLOv8m + LLaVA-1.5-7B + Hybrid Najm Liability**  
Classification: `rear_end` | `side_collision` only

> After running Cell 1, go to **Runtime → Restart session**, then run from Cell 2.

In [ ]:
# CELL 1 — Install libraries
!pip uninstall -y transformers tokenizers
!pip install -q --upgrade transformers tokenizers
!pip install -q accelerate bitsandbytes ultralytics
!pip install -q Pillow opencv-python-headless gradio matplotlib

import transformers
print(f'transformers: {transformers.__version__}')
print('Done — now: Runtime -> Restart session, then run from Cell 2')

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 64.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
transformers: 5.9.0
Done — now: Runtime -> Restart session, then run from Cell 2


## Cell 2 — Imports + GPU check

In [ ]:
import torch
import numpy as np
import cv2
import json
import re
import os
import warnings
import math
from PIL import Image
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ultralytics import YOLO
import transformers

warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'transformers : {transformers.__version__}')
print(f'torch        : {torch.__version__}')
if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU          : {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('No GPU detected — Runtime -> Change runtime type -> T4 GPU')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
transformers : 5.9.0
torch        : 2.11.0+cu128
GPU          : Tesla T4 (15.6 GB)


## Cell 3 — Load YOLOv8m

In [ ]:
print('Loading YOLOv8m...')
yolo = YOLO('yolov8m.pt')

VEHICLE_IDS = {2: 'Car', 3: 'Motorcycle', 5: 'Bus', 7: 'Truck', 1: 'Bicycle'}
COLOR_A = (0, 210, 255)   # Party 1 — yellow-ish, BGR
COLOR_B = (0, 140, 255)   # Party 2 — orange,     BGR

print('YOLOv8m ready')

Loading YOLOv8m...
YOLOv8m ready


## Cell 4 — Load LLaVA-1.5-7B (4-bit)

In [ ]:
from transformers import (
    AutoProcessor,
    LlavaForConditionalGeneration,
    BitsAndBytesConfig,
)

MODEL_ID = 'llava-hf/llava-1.5-7b-hf'
print(f'Loading {MODEL_ID}...')

processor   = AutoProcessor.from_pretrained(MODEL_ID)
llava_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type='nf4',
    ),
    device_map='auto',
    low_cpu_mem_usage=True,
)
llava_model.eval()

if device == 'cuda':
    print(f'LLaVA ready — {torch.cuda.memory_allocated()/1e9:.1f} GB used')
else:
    print('LLaVA ready (CPU mode)')

Loading llava-hf/llava-1.5-7b-hf...


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/70.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA ready — 4.4 GB used


## Cell 5 — Accident categories + geometry classifier

Only two accident types: `rear_end` and `side_collision`.  
The classifier uses a **multi-factor evidence scoring** system:
- Angle between vehicle centers
- Vertical / horizontal overlap ratios
- Contact zone ratio on each vehicle (front / side / rear)
- Lane-line detection via Hough Lines (strong side_collision signal)
- IoU of bounding boxes

All factors are combined into a sigmoid probability → highest wins.

In [ ]:
NAJM_VALID = [0, 25, 50, 75, 100]

ACCIDENT_CATEGORIES = {
    'rear_end': {
        'name_en': 'Rear-end collision',
        'viol_fault': [
            'Failed to maintain safe following distance',
            'Did not brake in time',
        ],
        'viol_other': ['No violation — was hit from behind'],
    },
    'side_collision': {
        'name_en': 'Side collision / lane-change sideswipe',
        'viol_fault': [
            'Changed lane without ensuring safety',
            'Entered another vehicle lane or space',
        ],
        'viol_other': [
            'Lower or no violation depending on movement evidence',
        ],
    },
}

# ── Geometry helpers ──────────────────────────────────────────

def snap_to_najm(pct: float) -> int:
    return min(NAJM_VALID, key=lambda v: abs(v - pct))

def snap_pair_to_najm(party1_raw: float) -> tuple:
    p1 = snap_to_najm(float(np.clip(round(party1_raw), 0, 100)))
    return int(p1), int(100 - p1)

def clamp(v, lo, hi):
    return max(lo, min(hi, v))

def _iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    union = (ax2 - ax1) * (ay2 - ay1) + (bx2 - bx1) * (by2 - by1) - inter
    return inter / union if union > 0 else 0.0

def center_of(v):
    if 'center' in v:
        return v['center']
    x1, y1, x2, y2 = v['bbox']
    return (x1 + x2) // 2, (y1 + y2) // 2

def bbox_dims(b):
    x1, y1, x2, y2 = b
    return max(1, x2 - x1), max(1, y2 - y1)

def vertical_overlap_ratio(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    overlap = max(0, min(ay2, by2) - max(ay1, by1))
    _, ah = bbox_dims(a)
    _, bh = bbox_dims(b)
    return overlap / max(1, min(ah, bh))

def horizontal_overlap_ratio(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    overlap = max(0, min(ax2, bx2) - max(ax1, bx1))
    aw, _ = bbox_dims(a)
    bw, _ = bbox_dims(b)
    return overlap / max(1, min(aw, bw))

def contact_zone_ratio(v1_bbox, contact_x):
    x1, _, x2, _ = v1_bbox
    w = max(1, x2 - x1)
    return clamp((contact_x - x1) / w, 0.0, 1.0)

def detect_lane_line_between(image_bgr, v1_bbox, v2_bbox):
    ax1, ay1, ax2, ay2 = v1_bbox
    bx1, by1, bx2, by2 = v2_bbox
    left_edge  = min(ax2, bx2)
    right_edge = max(ax1, bx1)
    if right_edge <= left_edge:
        return False
    top    = min(ay1, by1)
    bottom = max(ay2, by2)
    roi    = image_bgr[top:bottom, left_edge:right_edge]
    if roi.size == 0:
        return False
    gray  = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(
        edges, 1, np.pi / 180,
        threshold=40, minLineLength=60, maxLineGap=15,
    )
    if lines is None:
        return False
    for line in lines:
        x1l, y1l, x2l, y2l = line[0]
        angle = abs(math.degrees(math.atan2(y2l - y1l, x2l - x1l))) % 180
        if 60 <= angle <= 120:
            return True
    return False

def _find_rear_fault(v0, v1, dmg0, dmg1, c0, c1):
    if dmg0 == 'front' and dmg1 == 'rear':
        return 0
    if dmg0 == 'rear' and dmg1 == 'front':
        return 1
    if dmg0 == 'front':
        return 0
    if dmg1 == 'front':
        return 1
    if abs(c0[1] - c1[1]) > 20:
        return 0 if c0[1] > c1[1] else 1
    return None


def _find_side_fault(v0, v1, dmg0, dmg1, context):
    p1_stationary = context.get('party1_stationary', False)
    p2_stationary = context.get('party2_stationary', False)

    if p1_stationary and not p2_stationary:
        return 1
    if p2_stationary and not p1_stationary:
        return 0

    if dmg0 == 'right' and dmg1 == 'left':
        return 0
    if dmg0 == 'left' and dmg1 == 'right':
        return 1

    if dmg0 in ['left', 'right'] and dmg1 not in ['left', 'right']:
        return 0
    if dmg1 in ['left', 'right'] and dmg0 not in ['left', 'right']:
        return 1

    return None


def estimate_car_orientation(bbox):
    x1, y1, x2, y2 = bbox
    w = max(1, x2 - x1)
    h = max(1, y2 - y1)
    ratio = w / h
    if ratio > 1.6:
        return 'side_view'
    elif ratio < 1.2:
        return 'rear_front_view'
    else:
        return 'unknown'


# ── Core geometry classifier ──────────────────────────────────

def classify_accident_geometry(vehicles, image_bgr=None, context=None):
    context = context or {}

    if len(vehicles) < 2:
        return {
            'accident_type':     'side_collision',
            'fault_party_index': None,
            'raw_fault_pct':     50.0,
            'confidence':        20,
            'reason':            'Less than two vehicles detected.',
            'evidence_scores':   {},
        }

    v0, v1 = vehicles[0], vehicles[1]
    c0, c1 = center_of(v0), center_of(v1)
    dx     = c1[0] - c0[0]
    dy     = c1[1] - c0[1]
    angle  = abs(math.degrees(math.atan2(dy, dx))) % 180
    v_ovlp = vertical_overlap_ratio(v0['bbox'], v1['bbox'])
    h_ovlp = horizontal_overlap_ratio(v0['bbox'], v1['bbox'])
    iou    = _iou(v0['bbox'], v1['bbox'])

    # ── damage_side: يحدد مكان الضرر بناءً على نقطة التماس الفعلية ──
    def damage_side(va_bbox, vb_bbox):
        ax1, ay1, ax2, ay2 = va_bbox
        bx1, by1, bx2, by2 = vb_bbox

        inter_x1 = max(ax1, bx1)
        inter_y1 = max(ay1, by1)
        inter_x2 = min(ax2, bx2)
        inter_y2 = min(ay2, by2)

        has_overlap = (inter_x2 > inter_x1) and (inter_y2 > inter_y1)

        if has_overlap:
            contact_x = (inter_x1 + inter_x2) / 2
            contact_y = (inter_y1 + inter_y2) / 2
        else:
            contact_x = max(ax1, min(ax2, (bx1 + bx2) / 2))
            contact_y = max(ay1, min(ay2, (by1 + by2) / 2))

        aw = max(1, ax2 - ax1)
        ah = max(1, ay2 - ay1)

        rel_x = (contact_x - ax1) / aw
        rel_y = (contact_y - ay1) / ah

        sides = {
            'left':  rel_x,
            'right': 1.0 - rel_x,
            'front': rel_y,
            'rear':  1.0 - rel_y,
        }
        return min(sides, key=sides.get)

    dmg0 = damage_side(v0['bbox'], v1['bbox'])
    dmg1 = damage_side(v1['bbox'], v0['bbox'])


    # ── Scoring System ────────────────────────────────────────
    score = 0.0
    notes = []

    # 1. زاوية بين مراكز السيارات
    if angle <= 20 or angle >= 160:
        score += 2.0
        notes.append(f'Inline angle ({angle:.0f}deg) → rear_end +2')
    elif 30 <= angle <= 150:
        score -= 2.5
        notes.append(f'Non-inline angle ({angle:.0f}deg) → side_collision +2.5')

    # 2. Vertical overlap
    if v_ovlp >= 0.55:
        score += 1.5
        notes.append(f'High vertical overlap ({v_ovlp:.2f}) → rear_end +1.5')
    elif v_ovlp <= 0.25:
        score -= 2.0
        notes.append(f'Low vertical overlap ({v_ovlp:.2f}) → side_collision +2')

    # 3. Horizontal gap inline configuration
    left_idx  = 0 if c0[0] <= c1[0] else 1
    right_idx = 1 - left_idx
    left_v    = vehicles[left_idx]
    right_v   = vehicles[right_idx]
    gap_x     = max(0, right_v['bbox'][0] - left_v['bbox'][2])
    avg_w     = (bbox_dims(v0['bbox'])[0] + bbox_dims(v1['bbox'])[0]) / 2

    if gap_x <= avg_w * 0.12 and (angle <= 25 or angle >= 155):
        score += 2.0
        notes.append('Close front-rear gap in inline config → rear_end +2')

    # 4. نمط الضرر — damage_side يحسم النتيجة
    rear_end_damage = (dmg0 in ['front', 'rear'] and dmg1 in ['front', 'rear'])
    side_damage     = (dmg0 in ['left', 'right'] or dmg1 in ['left', 'right'])

    if rear_end_damage and not side_damage:
        score += 2.5
        notes.append(f'Front/rear damage pattern (v0={dmg0}, v1={dmg1}) → rear_end +2.5')

    if side_damage:
        score -= 5.0
        notes.append(f'Side damage pattern (v0={dmg0}, v1={dmg1}) → side_collision +5')
        # إذا كلا السيارتين ضررهم جانبي → تأكيد إضافي
        if dmg0 in ['left', 'right'] and dmg1 in ['left', 'right']:
            score -= 3.0
            notes.append(f'Both vehicles side damage confirmed → side_collision +3')

    # 5. IoU
    if iou > 0.05:
        if angle <= 30 or angle >= 150:
            score += 1.5
            notes.append('IoU overlap in inline vehicles → rear_end +1.5')
        else:
            score -= 1.5
            notes.append('IoU overlap in non-inline vehicles → side_collision +1.5')

    # 6. Lane lines
    lane_between = False
    if image_bgr is not None:
        lane_between = detect_lane_line_between(image_bgr, v0['bbox'], v1['bbox'])
    if lane_between:
        score -= 3.0
        notes.append('Lane marking detected between vehicles → side_collision +3')

    # 7. Horizontal overlap
    if h_ovlp >= 0.45:
        score -= 2.0
        notes.append(f'High horizontal overlap ({h_ovlp:.2f}) → side_collision +2')

# 8. Car orientation من شكل الـ bbox
    orient0 = estimate_car_orientation(v0['bbox'])
    orient1 = estimate_car_orientation(v1['bbox'])

    if side_damage and (angle <= 25 or angle >= 155):
        if orient0 == 'side_view' or orient1 == 'side_view':
            score += 6.0
            notes.append(f'Side-view car detected (orient0={orient0}, orient1={orient1}) → rear_end +6')
        elif orient0 == 'rear_front_view' and orient1 == 'rear_front_view':
            score -= 4.0
            notes.append(f'Rear-view cars detected → side_collision +4')



    # ── تحديد النوع ──────────────────────────────────────────
    rear_prob = 1.0 / (1.0 + math.exp(-score * 0.45))

    if rear_prob >= 0.55:
        acc_type = 'rear_end'
        t        = (rear_prob - 0.55) / (1.0 - 0.55)
        raw_pct  = 60.0 + t * 40.0
        raw_pct  = clamp(raw_pct, 60.0, 100.0)
        conf     = int(clamp(rear_prob * 100, 55, 95))

        fault_idx = _find_rear_fault(v0, v1, dmg0, dmg1, c0, c1)

        if fault_idx is not None:
            reason = (
                f'Rear-end pattern (rear_prob={rear_prob:.2f}). '
                f'Damage: v0={dmg0}, v1={dmg1}. '
                f'Fault → Party {fault_idx + 1} (the rear vehicle).'
            )
        else:
            reason = (
                f'Rear-end pattern (rear_prob={rear_prob:.2f}). '
                f'Damage: v0={dmg0}, v1={dmg1}. '
                f'Cannot determine fault clearly — shared liability possible.'
            )

    else:
        acc_type  = 'side_collision'
        side_prob = 1.0 - rear_prob
        t         = (side_prob - 0.45) / (1.0 - 0.45)
        raw_pct   = 55.0 + t * 45.0
        raw_pct   = clamp(raw_pct, 55.0, 100.0)
        conf      = int(clamp(side_prob * 100, 55, 92))

        fault_idx = _find_side_fault(v0, v1, dmg0, dmg1, context)

        if fault_idx is not None:
            reason = (
                f'Side-collision pattern (side_prob={side_prob:.2f}). '
                f'Damage: v0={dmg0}, v1={dmg1}. '
                f'Fault → Party {fault_idx + 1}.'
            )
        else:
            reason = (
                f'Side-collision pattern (side_prob={side_prob:.2f}). '
                f'Damage: v0={dmg0}, v1={dmg1}. '
                f'Cannot determine fault clearly — shared liability possible.'
            )

    return {
        'accident_type':     acc_type,
        'fault_party_index': fault_idx,
        'raw_fault_pct':     round(raw_pct, 1),
        'confidence':        conf,
        'reason':            reason,
        'evidence_scores':   {
            'score':     round(score, 2),
            'rear_prob': round(rear_prob, 3),
            'dmg0':      dmg0,
            'dmg1':      dmg1,
            'notes':     notes,
        },
    }

print('Geometry classifier ready (rear_end | side_collision)')

Geometry classifier ready (rear_end | side_collision)


## Cell 6 — LLaVA helpers + hybrid liability

LLaVA is asked 4 questions per photo:
1. Accident type (A=rear_end / B=side_collision)
2. Severity (low / medium / high)
3. What the at-fault driver did wrong
4. 3 visual evidence observations

Liability is a **continuous blend** of geometry prior + LLaVA estimate, then snapped to Najm scale.

In [ ]:
PROMPT_TYPE = """\
Look carefully at this traffic accident image.
The YELLOW bounding box is Party 1. The ORANGE bounding box is Party 2.
YOLO detection summary: {yolo_summary}

Based ONLY on what you see, which accident type best describes this?

A) rear_end   — one vehicle hit the other from behind (inline, front-to-rear contact)
B) side_collision — one vehicle sideswiped or merged into the other (vehicles side-by-side, contact on the flank)

Important clues:
- If vehicles are lined up one behind the other -> choose A
- If vehicles are side by side and touching flanks -> choose B
- If a lane marking is visible between them -> choose B
- Damage location matters: rear bumper/trunk damage = A, door/pillar damage = B

Reply with ONLY one letter: A or B."""

PROMPT_SEVERITY = """\
Look at this traffic accident image.
Estimate the severity of damage visible. Reply with ONLY one word: low, medium, or high."""

PROMPT_WHAT = """\
Look at this traffic accident image.
In ONE sentence only, describe what the likely at-fault driver did wrong. Do not give percentages."""

PROMPT_EVIDENCE = """\
Look at this traffic accident image.
Give exactly 3 short visual observations that help identify what happened.
One observation per line. No numbers, no bullets."""

PROMPT_LIABILITY = """\
You are a traffic accident liability expert following Saudi Najm rules.
The YELLOW box is Party 1. The ORANGE box is Party 2.

Accident type confirmed: {acc_type}
Geometry analysis: {geo_json}

Based on the image and geometry:
1. Who is more at fault and why?
2. Estimate the raw liability percentage for Party 1 (0-100).
   Do NOT round to 0, 25, 50, 75, or 100 — give your best continuous estimate.

Return ONLY valid JSON (no markdown, no extra text):
{{"party1_raw": <number 0-100>, "party2_raw": <number 0-100>, "fault_party": <1 or 2>, "reason": "<short reason>"}}"""

TYPE_LETTER_MAP = {'a': 'rear_end', 'b': 'side_collision'}

def ask_llava(image: Image.Image, question: str, max_tokens: int = 500) -> str:
    prompt  = f'USER: <image>\n{question}\nASSISTANT:'
    inputs  = processor(text=prompt, images=image, return_tensors='pt').to(device, torch.float16)
    with torch.no_grad():
        out = llava_model.generate(
            **inputs, max_new_tokens=max_tokens,
            do_sample=False, repetition_penalty=1.1,
        )
    answer = processor.decode(
        out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    ).strip()
    del inputs, out
    if device == 'cuda':
        torch.cuda.empty_cache()
    return answer

def _extract_type_letter(text: str) -> str:
    t = text.strip().lower()
    if t and t[0] in TYPE_LETTER_MAP:
        return t[0]
    m = re.search(r'(?<![a-z])([ab])(?![a-z])', t)
    if m:
        return m.group(1)
    if any(w in t for w in ['rear', 'behind', 'front-to-rear', 'bumper']):
        return 'a'
    return 'b'

def _extract_severity(text: str) -> str:
    t = text.lower()
    if 'high' in t: return 'high'
    if 'low'  in t: return 'low'
    return 'medium'

def _extract_evidence(text: str) -> list:
    lines = [
        l.strip().lstrip('*-*0123456789.) ')
        for l in text.strip().split('\n') if l.strip()
    ]
    return [l for l in lines if len(l) > 8][:3] or ['See annotated image for details']

def _extract_liability_json(text: str):
    try:
        m    = re.search(r'\{.*\}', text, flags=re.S)
        data = json.loads(m.group(0) if m else text)
        p1   = float(data.get('party1_raw', 50))
        p2   = float(data.get('party2_raw', 100 - p1))
        if abs((p1 + p2) - 100) > 8:
            p2 = 100 - p1
        return {
            'party1_raw':  clamp(round(p1, 1), 0, 100),
            'party2_raw':  clamp(round(p2, 1), 0, 100),
            'fault_party': int(data.get('fault_party', 0) or 0),
            'reason':      str(data.get('reason', '')).strip()[:240],
        }
    except Exception:
        return None

def diagnose(annotated_pil: Image.Image, yolo_summary: str, geo_type: str) -> dict:
    print('   LLaVA 1/4 — accident type classification...')
    raw_type   = ask_llava(annotated_pil, PROMPT_TYPE.format(yolo_summary=yolo_summary), max_tokens=15)
    llava_type = TYPE_LETTER_MAP[_extract_type_letter(raw_type)]
    print(f'   -> raw="{raw_type.strip()}" -> llava_type={llava_type}')

    print('   LLaVA 2/4 — severity...')
    raw_sev  = ask_llava(annotated_pil, PROMPT_SEVERITY, max_tokens=8)
    severity = _extract_severity(raw_sev)
    print(f'   -> "{raw_sev.strip()}" -> {severity}')

    print('   LLaVA 3/4 — what happened...')
    what_happened = ask_llava(annotated_pil, PROMPT_WHAT, max_tokens=70).strip()
    print(f'   -> {what_happened[:90]}')

    print('   LLaVA 4/4 — visual evidence...')
    raw_ev   = ask_llava(annotated_pil, PROMPT_EVIDENCE, max_tokens=90)
    evidence = _extract_evidence(raw_ev)

    return {
        'llava_type':    llava_type,
        'severity':      severity,
        'what_happened': what_happened,
        'evidence':      evidence,
    }

def model_liability_estimate(annotated_pil: Image.Image, geo: dict):
    geo_safe = {k: v for k, v in geo.items() if k not in ['evidence_scores']}
    raw = ask_llava(
        annotated_pil,
        PROMPT_LIABILITY.format(
            acc_type=geo['accident_type'],
            geo_json=json.dumps(geo_safe, ensure_ascii=False),
        ),
        max_tokens=140,
    )
    return _extract_liability_json(raw)

def resolve_accident_type(geo_type: str, geo_conf: int, llava_type: str) -> tuple:
    if geo_type == llava_type:
        combined = min(99, int(geo_conf * 1.15))
        return geo_type, combined, 'geometry+llava_agree'
    llava_conf = 70
    if geo_conf >= llava_conf:
        return geo_type, geo_conf, 'geometry_wins'
    else:
        return llava_type, llava_conf, 'llava_wins'

def _hybrid_liability_raw(geo: dict, model_pct) -> tuple:
    fault_idx = geo.get('fault_party_index')
    geo_raw   = geo.get('raw_fault_pct', 75.0)
    acc_type  = geo.get('accident_type', 'side_collision')

    if model_pct:
        p1m, p2m = model_pct['party1_raw'], model_pct['party2_raw']
        if abs(p1m - p2m) >= 15:
            if fault_idx == 0:   geo_p1 = geo_raw
            elif fault_idx == 1: geo_p1 = 100.0 - geo_raw
            else:                geo_p1 = 50.0
            blended = 0.50 * geo_p1 + 0.50 * p1m
            reason  = f'Blended geometry ({geo_p1:.1f}%) and model ({p1m:.1f}%) estimates. Accident: {acc_type}.'
            return round(blended, 1), round(100 - blended, 1), reason
        reason = f'Model near-equal ({p1m:.1f}%/{p2m:.1f}%); geometry prior used. Accident: {acc_type}.'
    else:
        reason = f'No model estimate; geometry prior used. Accident: {acc_type}.'

    if fault_idx == 0:   p1 = geo_raw
    elif fault_idx == 1: p1 = 100.0 - geo_raw
    else:                p1 = 50.0
    return round(p1, 1), round(100 - p1, 1), reason

def apply_liability(geo: dict, llava_type: str, annotated_pil: Image.Image) -> dict:
    final_type, type_conf, type_source = resolve_accident_type(
        geo['accident_type'], geo['confidence'], llava_type,
    )
    geo_for_liability = {**geo, 'accident_type': final_type}

    model_pct = None
    try:
        model_pct = model_liability_estimate(annotated_pil, geo_for_liability)
    except Exception as e:
        print(f'   Liability model warning: {e}')

    raw_a, raw_b, liability_reason = _hybrid_liability_raw(geo_for_liability, model_pct)
    final_a, final_b = snap_pair_to_najm(raw_a)

    fault_idx = 0 if final_a > final_b else (1 if final_b > final_a else None)

    category   = ACCIDENT_CATEGORIES[final_type]
    violations = []
    for idx in range(2):
        if fault_idx is None:    violations.append(category['viol_fault'])
        elif idx == fault_idx:   violations.append(category['viol_fault'])
        else:                    violations.append(category['viol_other'])

    print(f'   Accident type  : {final_type} (source={type_source})')
    if model_pct:
        print(f'   Model raw      : P1={model_pct["party1_raw"]}% | P2={model_pct["party2_raw"]}%')
    print(f'   Hybrid raw     : P1={raw_a}% | P2={raw_b}%')
    print(f'   Najm final     : P1={final_a}% | P2={final_b}%')
    print(f'   Reason         : {liability_reason}')

    return {
        'accident_type':     final_type,
        'name_en':           category['name_en'],
        'type_confidence':   type_conf,
        'type_source':       type_source,
        'fault_party_index': fault_idx,
        'final_pct_a':       final_a,
        'final_pct_b':       final_b,
        'base_pct_a':        raw_a,
        'base_pct_b':        raw_b,
        'viol_a':            violations[0],
        'viol_b':            violations[1],
        'liability_reason':  liability_reason,
    }

print('LLaVA helpers + hybrid liability ready')

LLaVA helpers + hybrid liability ready


## Cell 7 — YOLOv8 detection + annotation

In [ ]:
def detect_and_annotate(img: np.ndarray) -> tuple:
    h, w    = img.shape[:2]
    results = yolo(img, conf=0.40, verbose=False)[0]

    all_vehicles = []
    for box in results.boxes:
        cls_id = int(box.cls[0])
        if cls_id not in VEHICLE_IDS:
            continue
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        all_vehicles.append({
            'class':  VEHICLE_IDS[cls_id],
            'conf':   conf,
            'bbox':   (x1, y1, x2, y2),
            'center': ((x1 + x2) // 2, (y1 + y2) // 2),
            'area':   (x2 - x1) * (y2 - y1),
        })

    all_vehicles.sort(key=lambda v: v['area'], reverse=True)
    main_vehicles = all_vehicles[:2]
    annotated     = img.copy()
    colors        = [COLOR_A, COLOR_B]

    for i, v in enumerate(main_vehicles):
        col  = colors[i]
        x1, y1, x2, y2 = v['bbox']
        cv2.rectangle(annotated, (x1, y1), (x2, y2), col, 4)
        cv2.rectangle(annotated, (x1+4, y1+4), (x2-4, y2-4), col, 1)
        label = f'Party {i + 1}'
        font, fs, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.80, 2
        (tw, th), _ = cv2.getTextSize(label, font, fs, thick)
        lx = x1 + (x2 - x1) // 2 - tw // 2
        ly = y1 - 8 if y1 - 8 >= th + 4 else y1 + th + 8
        cv2.putText(annotated, label, (lx+1, ly+1), font, fs, (0,0,0),   thick+1)
        cv2.putText(annotated, label, (lx,   ly),   font, fs, col,       thick)
        v['color'] = col
        v['party'] = i + 1

    collisions = []
    if len(main_vehicles) == 2:
        va, vb = main_vehicles[0], main_vehicles[1]
        iou    = _iou(va['bbox'], vb['bbox'])
        dist   = math.hypot(va['center'][0]-vb['center'][0], va['center'][1]-vb['center'][1])
        avg_sz = math.sqrt((va['area'] + vb['area']) / 2)
        if iou > 0 or dist < avg_sz * 1.8:
            angle    = abs(math.degrees(math.atan2(
                vb['center'][1]-va['center'][1], vb['center'][0]-va['center'][0]))) % 180
            geo_type = _geo_label(angle, iou)
            _dashed(annotated, va['center'], vb['center'], (0, 0, 220), 3)
            mid = ((va['center'][0]+vb['center'][0])//2, (va['center'][1]+vb['center'][1])//2)
            cv2.circle(annotated, mid, 20, (0,0,200), -1)
            cv2.circle(annotated, mid, 20, (255,255,255), 2)
            cv2.putText(annotated, '!', (mid[0]-6, mid[1]+7),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255,255,255), 3)
            collisions.append({'va': va, 'vb': vb, 'iou': iou, 'type': geo_type})

    bar = np.zeros((48, w, 3), dtype=np.uint8)
    bar[:] = (15, 18, 28)
    status = (
        f'YOLOv8m | Vehicles: {len(main_vehicles)}/{len(all_vehicles)} | '
        + (f'Collision: {collisions[0]["type"]}' if collisions else 'No overlap detected')
    )
    cv2.putText(bar, status, (12, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.62, (232,184,75), 2)
    annotated = np.vstack([annotated, bar])
    return main_vehicles, collisions, annotated

def _geo_label(angle, iou):
    if iou >= 0.20:              return 'direct-overlap'
    if angle <= 25 or angle >= 155: return 'inline'
    if 65 <= angle <= 115:       return 'perpendicular'
    return 'angled'

def _dashed(img, p1, p2, color, thick):
    dist  = math.hypot(p2[0]-p1[0], p2[1]-p1[1])
    steps = max(1, int(dist / 18))
    for k in range(steps):
        t1 = k / steps
        t2 = min(1.0, (k + 0.55) / steps)
        s  = (int(p1[0]+t1*(p2[0]-p1[0])), int(p1[1]+t1*(p2[1]-p1[1])))
        e  = (int(p1[0]+t2*(p2[0]-p1[0])), int(p1[1]+t2*(p2[1]-p1[1])))
        cv2.line(img, s, e, color, thick)

print('detect_and_annotate ready')

detect_and_annotate ready


## Cell 8 — Report image builder

In [ ]:
def _to_bgr_array(img):
    if isinstance(img, Image.Image):
        arr = np.array(img.convert('RGB'))
        return cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    arr = np.array(img)
    if arr.ndim == 2:
        return cv2.cvtColor(arr, cv2.COLOR_GRAY2BGR)
    if arr.shape[2] == 4:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGBA2BGR)
    return arr

def build_report_image(report: dict, out_path: str = 'hasem_report.png', show: bool = False):
    annotated = _to_bgr_array(report['annotated_img'])
    pa, pb    = report['parties']
    fig       = plt.figure(figsize=(18, 14))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle(
        f"Hasem Accident Report  |  {report['report_id']}  |  {report['timestamp']}",
        fontsize=13, color='#e8b84b', fontweight='bold', y=0.99,
    )

    ax1 = fig.add_subplot(3, 1, 1)
    ax1.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax1.set_title(
        f'YOLOv8m  |  Accident: {report["name_en"]}  |  Severity: {str(report["severity"]).upper()}  |  Confidence: {report["type_confidence"]}%',
        color='#9ca3af', fontsize=11,
    )
    ax1.axis('off')

    ax2 = fig.add_subplot(3, 1, 2)
    ax2.set_facecolor('#161a22')
    ax2.axis('off')
    ax2.text(0.5, 0.97, report['name_en'], ha='center', va='top', transform=ax2.transAxes,
             fontsize=13, color='#e8b84b', fontweight='bold')
    ax2.text(0.5, 0.89, f'Average liability from {len(report.get("photo_results", []))} photo(s)',
             ha='center', va='top', transform=ax2.transAxes, fontsize=9, color='#4be88a')

    party_colors = ['#e8d84b', '#e8914b']
    for i, (p, pcol) in enumerate(zip([pa, pb], party_colors)):
        x       = 0.03 + i * 0.50
        final_p = int(p['final_pct'])
        ax2.text(x+0.22, 0.81, p['role'], ha='center', va='top', transform=ax2.transAxes,
                 fontsize=9.5, color=pcol, fontweight='bold')
        ax2.text(x+0.22, 0.74, p['vehicle'], ha='center', va='top', transform=ax2.transAxes,
                 fontsize=9, color='#dde4f0',
                 bbox=dict(boxstyle='round,pad=0.35', facecolor='#1e2330', edgecolor=pcol, linewidth=1.5))
        ax2.text(x+0.22, 0.64, f'Raw avg: {p["base_pct"]:.1f}%  ->  Najm: {final_p}%',
                 ha='center', va='top', transform=ax2.transAxes,
                 fontsize=9, color='#9ca3af', style='italic')
        ax2.text(x+0.22, 0.56, f'{final_p}%', ha='center', va='top', transform=ax2.transAxes,
                 fontsize=42, color=pcol, fontweight='bold')
        if final_p > 0:
            ax2.add_patch(mpatches.FancyBboxPatch(
                (x+0.02, 0.35), final_p/100*0.42, 0.07,
                boxstyle='round,pad=0.01', facecolor=pcol, edgecolor='none',
                transform=ax2.transAxes, alpha=0.85))
        for vi, v in enumerate(p['violations'][:3]):
            ax2.text(x+0.02, 0.28-vi*0.085, f'  {v}', ha='left', va='top',
                     transform=ax2.transAxes, fontsize=8.5, color='#9ca3af')

    ax3 = fig.add_subplot(3, 1, 3)
    ax3.set_facecolor('#111318')
    ax3.axis('off')
    verdict = (
        f"Accident: {report['name_en']}\n"
        f"{pa['vehicle']}  ->  {pa['final_pct']}% liability\n"
        f"{pb['vehicle']}  ->  {pb['final_pct']}% liability\n"
        f"What happened: {report.get('what_happened', '')}\n"
        f"Reason: {report.get('liability_reason', '')}"
    )
    ax3.text(0.5, 0.94, verdict, ha='center', va='top', transform=ax3.transAxes,
             fontsize=9, color='#e8b84b', multialignment='center',
             bbox=dict(boxstyle='round,pad=0.5', facecolor='#1a1f2a', edgecolor='#e8b84b', alpha=0.7))

    per_photo = report.get('photo_results', [])
    if per_photo:
        lines = [
            f"Photo {r['photo_no']}: {r['name_en']} (conf={r['type_confidence']}%) | "
            f"P1 {r['final_pct_a']}% / P2 {r['final_pct_b']}%"
            for r in per_photo[:5]
        ]
        ax3.text(0.02, 0.52, 'Per-photo results:\n' + '\n'.join(lines),
                 ha='left', va='top', transform=ax3.transAxes, fontsize=8.5, color='#9ca3af')
    else:
        ev = report.get('evidence', [])
        ax3.text(0.02, 0.52, 'Visual evidence:\n' + '\n'.join(f'  {e}' for e in ev[:3]),
                 ha='left', va='top', transform=ax3.transAxes, fontsize=8.5, color='#9ca3af')

    recs = report.get('recommendations', [])
    ax3.text(0.55, 0.52, 'Recommendations:\n' + '\n'.join(f'  {i+1}. {r}' for i,r in enumerate(recs[:3])),
             ha='left', va='top', transform=ax3.transAxes, fontsize=8.5, color='#4be88a')
    ax3.text(0.02, 0.11,
             f'Avg confidence: {report["type_confidence"]}%  |  Source: {report.get("type_source", "")}',
             ha='left', va='center', transform=ax3.transAxes, fontsize=9, color='#4b8be8')
    ax3.text(0.55, 0.11,
             f'YOLO geometry: {report["yolo_type"]}  |  Najm avg: {pa["final_pct"]}% / {pb["final_pct"]}%',
             ha='left', va='center', transform=ax3.transAxes, fontsize=8.5, color='#6b7280', style='italic')

    plt.tight_layout(rect=[0, 0, 1, 0.98])
    plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    if show:
        plt.show()
    plt.close(fig)
    print(f'Report saved -> {out_path}')
    return out_path

def show_report(report: dict):
    return build_report_image(report, out_path='hasem_report.png', show=True)

print('build_report_image + show_report ready')

build_report_image + show_report ready


## Cell 9 — Main pipeline (1–4 photos)

- Each photo is analyzed independently
- The accident type with the **highest average confidence** across all photos is selected
- Photos that disagree with the winning type are excluded from liability averaging
- Final liability is a continuous average → snapped to Najm scale

In [ ]:
def _normalize_input_image(img):
    if isinstance(img, Image.Image):
        return img.convert('RGB')
    return Image.fromarray(np.array(img)).convert('RGB')

def _best_accident_type(photo_results: list) -> tuple:
    scores = {}
    for r in photo_results:
        t, conf = r['accident_type'], r['type_confidence']
        if t not in scores:
            scores[t] = {'total_conf': 0, 'count': 0}
        scores[t]['total_conf'] += conf
        scores[t]['count']      += 1
    ranked = sorted(
        scores.items(),
        key=lambda kv: (kv[1]['total_conf'] / kv[1]['count'], kv[1]['count']),
        reverse=True,
    )
    best_type = ranked[0][0]
    best_conf = int(ranked[0][1]['total_conf'] / ranked[0][1]['count'])
    return best_type, best_conf

def _majority_severity(photo_results: list) -> str:
    order = {'low': 1, 'medium': 2, 'high': 3}
    vals  = [str(r.get('severity', 'medium')).lower() for r in photo_results]
    if not vals:
        return 'medium'
    nums = sorted(order.get(v, 2) for v in vals)
    mid  = nums[len(nums) // 2]
    return {1: 'low', 2: 'medium', 3: 'high'}[mid]

def analyze_accident(images: list, context: dict = None) -> dict:
    context = context or {}
    images  = [_normalize_input_image(img) for img in images if img is not None]
    if len(images) < 1:
        raise ValueError('Please upload at least 1 accident photo.')
    images = images[:4]

    print('=' * 60)
    print('  Hasem v2  |  YOLOv8m + LLaVA + Hybrid Najm')
    print('  Classification: rear_end | side_collision')
    print('=' * 60)

    photo_results = []
    for idx, pil_img in enumerate(images, start=1):
        print(f'\n========== Photo {idx}/{len(images)} ==========')
        print('Step 1 — YOLO detection...')
        main_arr = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        vehicles, collisions, annotated = detect_and_annotate(main_arr)
        yolo_type = collisions[0]['type'] if collisions else 'none'
        print(f'   {len(vehicles)} vehicle(s) | {len(collisions)} collision zone(s) | geo={yolo_type}')

        print('Step 2 — Geometry classification...')
        geo = classify_accident_geometry(vehicles, image_bgr=main_arr)
        print(f'   geo_type={geo["accident_type"]} | conf={geo["confidence"]}%')

        yolo_sum      = f'{len(vehicles)} vehicles, collision geometry: {yolo_type}, photo {idx} of {len(images)}'
        annotated_pil = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))

        print('Step 3 — LLaVA diagnosis...')
        diag = diagnose(annotated_pil, yolo_sum, geo['accident_type'])

        print('Step 4 — Hybrid liability...')
        liability = apply_liability(geo, diag['llava_type'], annotated_pil)

        photo_results.append({
            'photo_no':        idx,
            'vehicles_count':  len(vehicles),
            'collision_zones': len(collisions),
            'yolo_type':       yolo_type,
            'accident_type':   liability['accident_type'],
            'name_en':         liability['name_en'],
            'type_confidence': liability['type_confidence'],
            'type_source':     liability['type_source'],
            'severity':        diag.get('severity', 'medium'),
            'final_pct_a':     liability['final_pct_a'],
            'final_pct_b':     liability['final_pct_b'],
            'base_pct_a':      liability['base_pct_a'],
            'base_pct_b':      liability['base_pct_b'],
            'fault_party_index': liability['fault_party_index'],
            'liability_reason': liability['liability_reason'],
            'what_happened':   diag.get('what_happened', ''),
            'evidence':        diag.get('evidence', []),
            'annotated_img':   annotated_pil,
            'vehicles':        vehicles,
            'collisions':      collisions,
        })

    overall_type, avg_type_conf = _best_accident_type(photo_results)
    matching = [r for r in photo_results if r['accident_type'] == overall_type] or photo_results

    avg_raw_a        = float(np.mean([r['base_pct_a'] for r in matching]))
    final_a, final_b = snap_pair_to_najm(avg_raw_a)
    avg_base_a       = round(avg_raw_a, 1)
    avg_base_b       = round(100 - avg_raw_a, 1)
    severity         = _majority_severity(photo_results)
    category         = ACCIDENT_CATEGORIES[overall_type]
    fault_idx        = 0 if final_a > final_b else (1 if final_b > final_a else None)

    def resolve_vehicle_name(ctx_key, yolo_idx, fallback):
        name = context.get(ctx_key, '').strip()
        if not name:
            hit = next((r for r in photo_results if len(r['vehicles']) > yolo_idx), None)
            if hit:
                name = hit['vehicles'][yolo_idx].get('class', fallback)
        return name or fallback

    va_name = resolve_vehicle_name('vehicle1', 0, 'Party 1 Vehicle')
    vb_name = resolve_vehicle_name('vehicle2', 1, 'Party 2 Vehicle')

    if fault_idx is None:
        pa_role, pb_role = 'Party 1 — Shared / Undetermined', 'Party 2 — Shared / Undetermined'
    else:
        pa_role = 'Party 1 — At-fault party' if fault_idx == 0 else 'Party 1 — Other party'
        pb_role = 'Party 2 — At-fault party' if fault_idx == 1 else 'Party 2 — Other party'

    violations = []
    for i in range(2):
        if fault_idx is None:  violations.append(category['viol_fault'])
        elif i == fault_idx:   violations.append(category['viol_fault'])
        else:                  violations.append(category['viol_other'])

    pa = {'role': pa_role, 'vehicle': va_name,
          'final_pct': final_a, 'base_pct': avg_base_a, 'violations': violations[0]}
    pb = {'role': pb_role, 'vehicle': vb_name,
          'final_pct': final_b, 'base_pct': avg_base_b, 'violations': violations[1]}

    representative = max(matching, key=lambda r: r['type_confidence'])
    try:
        recs_raw = ask_llava(
            representative['annotated_img'],
            f'Accident: {category["name_en"]}, severity: {severity}. '
            f'Give 3 short numbered recommendations for the parties involved.',
            max_tokens=160,
        )
        recs = [l.strip().lstrip('123.-) ') for l in recs_raw.split('\n') if l.strip()][:3]
    except Exception:
        recs = []
    recs = recs or [
        'Document the accident with traffic police immediately',
        'Contact Najm / insurance within 24 hours',
        'Preserve all photos and witness information',
    ]

    sources         = [r['type_source'] for r in matching]
    dominant_source = max(set(sources), key=sources.count)
    reason = (
        f"Final type '{overall_type}' selected with avg confidence {avg_type_conf}% "
        f"from {len(matching)}/{len(photo_results)} matching photo(s). "
        f"Per-photo P1 raw: "
        + ', '.join(f"P{r['photo_no']}={r['base_pct_a']:.1f}%" for r in matching)
        + f". Average raw={avg_base_a}%, Najm final={final_a}%/{final_b}%."
    )

    report = {
        'report_id':       f"HAD-{datetime.now().strftime('%Y')}-{np.random.randint(10000,99999)}",
        'timestamp':       datetime.now().strftime('%Y-%m-%d %H:%M'),
        'model':           'YOLOv8m + LLaVA-1.5-7B + Hybrid Najm v2',
        'accident_type':   overall_type,
        'name_en':         category['name_en'],
        'type_confidence': avg_type_conf,
        'type_source':     dominant_source,
        'severity':        severity,
        'parties':         [pa, pb],
        'what_happened':   representative.get('what_happened', ''),
        'evidence':        [e for r in photo_results for e in r.get('evidence', [])][:3],
        'recommendations': recs,
        'yolo_type':       ', '.join(r['yolo_type'] for r in photo_results),
        'liability_reason': reason,
        'annotated_img':   representative['annotated_img'],
        'annotated_image': representative['annotated_img'],
        'context':         context,
        'photo_results':   photo_results,
    }
    build_report_image(report, out_path='hasem_report.png')
    return report

print('Main pipeline ready: 1-4 photos | rear_end | side_collision')

Main pipeline ready: 1-4 photos | rear_end | side_collision


## Cell 10 — Quick test (Google Colab)

In [ ]:
# from google.colab import files

#  print('Upload 1-4 accident images...')
#  uploaded = files.upload()

#  if uploaded:
#      imgs   = [Image.open(name).convert('RGB') for name in list(uploaded.keys())[:4]]
#      report = analyze_accident(
#          images=imgs,
#          context={
#              'location': 'King Fahad Road',
#              'weather':  'Clear',
#              'roadType': 'Road',
#              'vehicle1': 'Party 1 Vehicle',
#              'vehicle2': 'Party 2 Vehicle',
#              'description': '',
#          },
#      )
#      show_report(report)
#  else:
#      print('No images uploaded.')

##CELL 11 — Flask API + Ngrok Integration for VS Code Communication

In [ ]:
!pip install -q flask flask-cors pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3Dcww0LkWJTRWk7Kk3AFC6xWZlS_2j6fR2W4296X9ieiNjiBd")

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
import base64
from PIL import Image
from io import BytesIO

app = Flask(__name__)
CORS(app)

@app.route("/analyze", methods=["POST"])
def analyze_api():
    try:
        files = request.files.getlist("images")

        if not files:
            return jsonify({"error": "No images uploaded"}), 400

        imgs = []

        for file in files[:4]:
            img = Image.open(file.stream).convert("RGB")
            imgs.append(img)

        report = analyze_accident(
            images=imgs,
            context={
                "location": "Saudi Arabia",
                "weather": "Clear",
                "roadType": "Road",
                "vehicle1": "Party 1 Vehicle",
                "vehicle2": "Party 2 Vehicle",
                "description": request.form.get("prompt", "")
            }
        )

        build_report_image(report, out_path="hasem_report.png")

        annotated_only_path = "annotated_detection.png"

        report["annotated_img"].save(annotated_only_path)

        with open("annotated_detection.png", "rb") as f:
            report_image_base64 = base64.b64encode(f.read()).decode("utf-8")

        return jsonify({
            "accident_type": report["name_en"],
            "liability_percentage": report["parties"][0]["final_pct"],
            "type_confidence": report["type_confidence"],
            "parties": report["parties"],
            "what_happened": report["what_happened"],
            "evidence": report["evidence"],
            "liability_reason": report["liability_reason"],
            "report_image_base64": report_image_base64
        })

    except Exception as e:
        print("API ERROR:", e)
        return jsonify({"error": str(e)}), 500


public_url = ngrok.connect(5000)
print("API URL:", public_url)

app.run(port=5000)